In [1]:
!python --version

Python 3.12.14


In [9]:
!pip install -Uq pandas requests

In [10]:
import pandas as pd
import numpy as np
import requests

In [19]:
url = "https://www-nds.iaea.org/amdc/ame2020/mass_1.mas20.txt"

response = requests.get(url)

with open("mass.mas20", "wb") as f:
    f.write(response.content)

In [21]:

fname = "mass.mas20"

HEADER_LINES = 36   # check: the first data line should be the neutron (Z=0, A=1)

colspecs = [(1,4),(4,9),(9,14),(14,19),(20,23),(23,27),
            (28,42),(42,54),(54,67),(68,78),
            (79,81),(81,94),(94,105),
            (106,109),(110,123),(123,135)]
names = ["N_Z","N","Z","A","El","Orig",
         "MassExcess","MassExcess_unc",
         "BE_per_A","BE_per_A_unc",
         "BetaType","BetaDecay","BetaDecay_unc",
         "AM_int","AM_frac","AM_unc"]

df = pd.read_fwf(fname, colspecs=colspecs, names=names,
                 skiprows=HEADER_LINES, dtype=str, keep_default_na=False)



df.head()

,N_Z,N,Z,A,El,Orig,MassExcess,MassExcess_unc,BE_per_A,BE_per_A_unc,BetaType,BetaDecay,BetaDecay_unc,AM_int,AM_frac,AM_unc
0,1,1,0,1,n,,8071.31806,0.00044,0.0,0.0,B-,782.3470,0.0004,1,008664.91590,0.00047
1,-1,0,1,1,H,,7288.971064,0.000013,0.0,0.0,B-,*,,1,007825.031898,0.000014
2,0,1,1,2,H,,13135.722895,0.000015,1112.2831,0.0002,B-,*,,2,014101.777844,0.000015
3,1,2,1,3,H,,14949.81090,0.00008,2827.2654,0.0003,B-,18.59202,0.00006,3,016049.28132,0.00008
4,-1,1,2,3,He,,14931.21888,0.00006,2572.68044,0.00015,B-,-13736#,2000#,3,016029.32197,0.00006


In [22]:
def to_num(s):
    """Return (value, is_estimated). '*' or blank -> NaN."""
    s = s.strip()
    if s in ("", "*"):
        return np.nan, False
    est = "#" in s
    return float(s.replace("#", ".")), est

# numeric columns; also record which ones are estimated (#)
num_cols = ["MassExcess","MassExcess_unc","BE_per_A","BE_per_A_unc",
            "BetaDecay","BetaDecay_unc","AM_frac","AM_unc"]
for c in num_cols:
    vals = df[c].map(to_num)
    df[c] = vals.map(lambda t: t[0])
    df[c + "_est"] = vals.map(lambda t: t[1])

for c in ["N_Z","N","Z","A","AM_int"]:
    df[c] = pd.to_numeric(df[c].str.strip(), errors="coerce")

# full atomic mass in micro-u
df["AtomicMass_uu"] = df["AM_int"] * 1e6 + df["AM_frac"]
df["El"] = df["El"].str.strip()

df = df.drop(columns=["AM_int","AM_frac"])
df.head(12)


,N_Z,N,Z,A,El,Orig,MassExcess,MassExcess_unc,BE_per_A,BE_per_A_unc,...,AM_unc,MassExcess_est,MassExcess_unc_est,BE_per_A_est,BE_per_A_unc_est,BetaDecay_est,BetaDecay_unc_est,AM_frac_est,AM_unc_est,AtomicMass_uu
0,1,1,0,1,n,,8071.318060,0.000440,0.00000,0.00000,...,0.000470,False,False,False,False,False,False,False,False,1.008665e+06
1,-1,0,1,1,H,,7288.971064,0.000013,0.00000,0.00000,...,0.000014,False,False,False,False,False,False,False,False,1.007825e+06
2,0,1,1,2,H,,13135.722895,0.000015,1112.28310,0.00020,...,0.000015,False,False,False,False,False,False,False,False,2.014102e+06
3,1,2,1,3,H,,14949.810900,0.000080,2827.26540,0.00030,...,0.000080,False,False,False,False,False,False,False,False,3.016049e+06
4,-1,1,2,3,He,,14931.218880,0.000060,2572.68044,0.00015,...,0.000060,False,False,False,False,True,True,False,False,3.016029e+06
5,-3,0,3,3,Li,-pp,28667.000000,2000.000000,-2267.00000,667.00000,...,2147.000000,True,True,True,True,False,False,True,True,3.030775e+06
6,2,3,1,4,H,-n,24621.129000,100.000000,1720.44910,25.00000,...,107.354000,False,False,False,False,False,False,False,False,4.026432e+06
7,0,2,2,4,He,,2424.915870,0.000150,7073.91560,0.00020,...,0.000160,False,False,False,False,False,False,False,False,4.002603e+06
8,-2,1,3,4,Li,-p,25323.190000,212.132000,1153.76030,53.03300,...,227.733000,False,False,False,False,False,False,False,False,4.027186e+06
9,3,4,1,5,H,-nn,32892.447000,89.443000,1336.35920,17.88850,...,96.020000,False,False,False,False,False,False,False,False,5.035311e+06


In [23]:
df.to_csv("ame2020_mass.csv", index=False)

In [24]:
df.columns

Index(['N_Z', 'N', 'Z', 'A', 'El', 'Orig', 'MassExcess', 'MassExcess_unc',
       'BE_per_A', 'BE_per_A_unc', 'BetaType', 'BetaDecay', 'BetaDecay_unc',
       'AM_unc', 'MassExcess_est', 'MassExcess_unc_est', 'BE_per_A_est',
       'BE_per_A_unc_est', 'BetaDecay_est', 'BetaDecay_unc_est', 'AM_frac_est',
       'AM_unc_est', 'AtomicMass_uu'],
      dtype='str')

In [56]:
import pandas as pd

COLUMNS = ['N_Z', 'N', 'Z', 'A', 'MassExcess', 'BE_per_A']
df = pd.read_csv("ame2020_mass.csv", usecols=COLUMNS)
df.head()

,N_Z,N,Z,A,MassExcess,BE_per_A
0,1,1,0,1,8071.318060,0.00000
1,-1,0,1,1,7288.971064,0.00000
2,0,1,1,2,13135.722895,1112.28310
3,1,2,1,3,14949.810900,2827.26540
4,-1,1,2,3,14931.218880,2572.68044


In [59]:
import numpy as np

_MAGIC_NUMBERS = [0, 2, 8, 20, 28, 50, 82, 126, 184, np.inf]

def delta_nucleons(nucleon_number)-> int:
    """
    Calculate the number of outer shell nucleons for a given nucleon number.

    Args:
        nucleon_number (int): the number of nucleons (either protons or neutrons).

    Returns:
        int: The number of nucleons outter or missing from the closest shell.
    
    Notes:
        Magic numbers 126 (protons) and 184 (neutrons) are theoretical and 
        have not been experimentally confirmed.
        For real nuclei, 295 Og is the heaviest known isotope, with Z=118 and N=177.
        For superheavy nuclei, results beyond Z=82 or N=126 depend on these predictions.
    """
    assert isinstance(nucleon_number, int) and nucleon_number >= 0, "nucleon_number must be a non-negative integer"

    lower = max([m for m in _MAGIC_NUMBERS if m <= nucleon_number])
    upper = min([m for m in _MAGIC_NUMBERS if m > nucleon_number])
    return int(min(nucleon_number - lower, upper - nucleon_number))   


df['DeltaZ'] = df['Z'].apply(delta_nucleons)
df['DeltaN'] = df['N'].apply(delta_nucleons)

df['DeltaA'] = df['DeltaZ'] + df['DeltaN']


In [62]:
# even-odd nuclei
def even_odd(n):
    """
    Determine if a number is even or odd.

    Args:
        n (int): The number to check.

    Returns:
        int: 0 if the number is even, 1 if the number is odd.
    """
    return n % 2
    
df['EvenOddZ'] = df['Z'].apply(even_odd)
df['EvenOddN'] = df['N'].apply(even_odd)

In [67]:
df['A^2/3']  = df['A']**(2/3)
df['Z(Z-1)/A^1/3']  = df['Z']*(df['Z']-1.0)/df['A']**(1/3)
df['(N-Z)^2/A']  = (df['N_Z'])**2/df['A']


In [68]:
df.head()

,N_Z,N,Z,A,MassExcess,BE_per_A,DeltaZ,DeltaN,DeltaA,EvenOddZ,EvenOddN,A^2/3,Z(Z-1)/A^1/3,(N-Z)^2/A
0,1,1,0,1,8071.318060,0.00000,0,1,1,0,1,1.000000,-0.000000,1.000000
1,-1,0,1,1,7288.971064,0.00000,1,0,1,1,0,1.000000,0.000000,1.000000
2,0,1,1,2,13135.722895,1112.28310,1,1,2,1,1,1.587401,0.000000,0.000000
3,1,2,1,3,14949.810900,2827.26540,1,0,1,1,0,2.080084,0.000000,0.333333
4,-1,1,2,3,14931.218880,2572.68044,0,1,1,0,1,2.080084,1.386723,0.333333


In [69]:
df.to_csv('dataset_nuclei_mass.csv', index=False)